# Data analysis of the original survey data provided by IBM

## Setup

In [3]:
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

/Users/melsh/opt/anaconda3/envs/DBLDataChallenge/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [4]:
DATA_FILE = Path('Transformed_SAX_Quality_of_Explanations_with_categories_transposed.xlsx')
EXCLUSIONS = None
KEEP_ALL_PARTICIPANTS = False

## Survey questions

In [5]:
CONSENT = (
    "I hereby give my consent to participate in this study, agreeing to the "
    "above conditions."
)

ITEMS = {
    "Completeness": [
        "The explanation provides all the details about the condition.",
        "The explanation is complete in covering relevant information.",
        "The explanation encompasses all aspects and outcomes related to the condition.",
    ],
    "Soundness": [
        "The explanation accurately reflects the properties of the condition.",
        "The explanation avoids assumptions not backed by data or process evidence.",
        "The explanation is consistent with the process.",
    ],
    "Causability": [
        "The explanation encloses what causes the condition.",
        "The explanation is about why the condition occurred.",
        "The sequence of events leading to the condition is clearly outlined.",
    ],
    "Clarity": [
        "The explanation is presented in an unambiguous and straightforward manner.",
        "I did not need any clarifications to interpret the explanation.",
        "The explanation can only have one interpretation.",
    ],
    "Compactness": [
        "The explanation is short.",
        "The explanation does not include irrelevant information.",
        "The key elements of the condition are succinctly presented.",
    ],
    "Comprehensibility": [
        "The explanation aligns with my perception on how the condition occurred.",
        "The explanation is easy to understand.",
        "The explanation matches my level of expertise and knowledge of the process.",
    ],
    "Curiosity": [
        "I am curious to know what explains the condition.",
        "I am eager to understand why the condition occurred.",
        "I am intrigued to find out why some other condition did not occur instead.",
    ],
    "Trust": [
        "I have confidence in the explanations generated by the LLM tool.",
        "I find the LLM tool to be highly reliable and accurate consistently.",
        "I feel secure in relying on the LLM tool for correct explanations.",
    ],
}

In [6]:
CONSTRUCTS = {
    "Fidelity": ["Completeness", "Soundness", "Causability"],
    "Interpretability": ["Clarity", "Compactness", "Comprehensibility"],
}

DOMAIN_NAMES = {1: "Pizza delivery", 2: "Parking fines", 3: "Loan approval"}

## Paper reported outcomes

In [7]:
PAPER_ALPHA = {
    "Completeness": 0.91,
    "Soundness": 0.63,
    "Causability": 0.69,
    "Clarity": 0.81,
    "Compactness": 0.49,
    "Comprehensibility": 0.64,
    "Curiosity": 0.92,
    "Trust": 0.90,
    "Fidelity": 0.81,
    "Interpretability": 0.80,
    "Compactness_without_item_3": 0.516,
}

PAPER_PCA = {
    "Completeness": ([0.919, 0.924, 0.914], 84.42),
    "Soundness": ([0.837, 0.606, 0.821], 58.05),
    "Causability": ([0.829, 0.804, 0.718], 61.61),
    "Clarity": ([0.846, 0.863, 0.841], 72.24),
    "Compactness": ([0.714, 0.804, 0.584], 49.88),
    "Comprehensibility": ([0.803, 0.667, 0.835], 59.53),
    "Curiosity": ([0.930, 0.955, 0.894], 85.88),
    "Trust": ([0.934, 0.869, 0.928], 82.94),
}

In [8]:
PAPER_MODELS = {
    ("Pizza delivery", "Fidelity", "ANOVA"): (4.44, 3.86, 2.42, "0.12", 0.051),
    ("Pizza delivery", "Interpretability", "ANOVA"): (3.98, 4.16, 0.42, "0.51", 0.009),
    ("Pizza delivery", "Fidelity", "ANCOVA"): (4.46, 3.84, 7.60, "<.001", 0.347),
    ("Pizza delivery", "Interpretability", "ANCOVA"): (4.01, 4.13, 5.08, "0.004", 0.262),
    ("Parking fines", "Fidelity", "ANOVA"): (3.23, 3.71, 2.71, "0.107", 0.057),
    ("Parking fines", "Interpretability", "ANOVA"): (3.63, 2.56, 15.30, "<.001", 0.254),
    ("Parking fines", "Fidelity", "ANCOVA"): (3.31, 3.62, 3.67, "0.019", 0.204),
    ("Parking fines", "Interpretability", "ANCOVA"): (3.73, 2.45, 9.84, "<.001", 0.407),
    ("Loan approval", "Fidelity", "ANOVA"): (4.05, 3.62, 1.97, "0.16", 0.042),
    ("Loan approval", "Interpretability", "ANOVA"): (3.31, 3.25, 0.058, "0.81", 0.001),
    ("Loan approval", "Fidelity", "ANCOVA"): (4.02, 3.65, 19.16, "<.001", 0.572),
    ("Loan approval", "Interpretability", "ANCOVA"): (3.27, 3.31, 10.56, "<.001", 0.424),
}

PAPER_COVARIATE_TERMS = {
    ("Pizza delivery", "Fidelity", "Trust"): (17.79, "<.001"),
    ("Pizza delivery", "Interpretability", "Trust"): (14.67, "<.001"),
    ("Parking fines", "Fidelity", "Trust"): (5.71, "0.021"),
    ("Parking fines", "Interpretability", "Curiosity"): (6.43, "0.015"),
    ("Parking fines", "Interpretability", "Trust"): (7.00, "0.011"),
    ("Loan approval", "Fidelity", "Curiosity"): (6.67, "0.013"),
    ("Loan approval", "Fidelity", "Trust"): (51.40, "<.001"),
    ("Loan approval", "Interpretability", "Trust"): (31.58, "<.001"),
}

## Functions

In [9]:
def normalize(value: object) -> str:
    text = str(value).replace("\u00a0", " ").replace("’", "'").strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text.rstrip(". ")

def find_column(columns: list[str], requested: str, aliases: tuple[str, ...] = ()) -> str:
    lookup = {normalize(column): column for column in columns}
    for candidate in (requested, *aliases):
        if normalize(candidate) in lookup:
            return lookup[normalize(candidate)]
    raise KeyError(f"Column not found: {requested}")

In [11]:
def cronbach_alpha(frame: pd.DataFrame) -> float:
    clean = frame.apply(pd.to_numeric, errors="coerce").dropna()
    k = clean.shape[1]
    if k < 2 or len(clean) < 2:
        return math.nan
    total_variance = clean.sum(axis=1).var(ddof=1)
    if not np.isfinite(total_variance) or total_variance <= 0:
        return math.nan
    return float(k / (k - 1) * (1 - clean.var(ddof=1).sum() / total_variance))

def pca_one_component(frame: pd.DataFrame) -> tuple[np.ndarray, float]:
    x = frame.apply(pd.to_numeric, errors="coerce").dropna()
    if (x.std(ddof=1) == 0).any():
        raise ValueError("A constant prevents correlation-based PCA.")
    correlation = x.corr().to_numpy()
    values, vectors = np.linalg.eigh(correlation)
    loading = vectors[:, -1] * np.sqrt(values[-1])
    if loading.sum() < 0:
        loading *= -1
    explained = float(100 * values[-1] / len(values))
    return loading, explained

In [12]:
def fit_ols(y: np.ndarray, x: np.ndarray) -> dict[str, object]:
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)
    n, p = x.shape
    if n <= p or np.linalg.matrix_rank(x) < p:
        raise ValueError("Model has too few observations or collinear predictors.")
    beta = np.linalg.lstsq(x, y, rcond=None)[0]
    fitted = x @ beta
    residual = y - fitted
    sse = float(residual @ residual)
    sst = float(((y - y.mean()) ** 2).sum())
    df_resid = n - p
    mse = sse / df_resid
    covariance = mse * np.linalg.inv(x.T @ x)
    df_model = p - 1
    ss_model = max(0.0, sst - sse)
    f_model = (ss_model / df_model) / mse
    return {
        "beta": beta,
        "covariance": covariance,
        "df_model": df_model,
        "df_resid": df_resid,
        "F": f_model,
        "p": float(stats.f.sf(f_model, df_model, df_resid)),
        "R2_eta_squared": ss_model / sst if sst > 0 else math.nan,
        "SSE": sse,
        "MSE": mse,
    }

def coefficient_test(model: dict[str, object], index: int) -> dict[str, float]:
    beta = np.asarray(model["beta"])
    covariance = np.asarray(model["covariance"])
    df = int(model["df_resid"])
    se = math.sqrt(float(covariance[index, index]))
    t_value = float(beta[index] / se)
    f_value = t_value**2
    return {
        "coefficient": float(beta[index]),
        "SE": se,
        "t": t_value,
        "F": f_value,
        "p": float(stats.f.sf(f_value, 1, df)),
        "partial_eta_squared": f_value / (f_value + df),
    }

In [13]:
def hedges_g(group1: np.ndarray, group2: np.ndarray) -> tuple[float, float]:
    a, b = np.asarray(group1, float), np.asarray(group2, float)
    pooled = math.sqrt(
        ((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1))
        / (len(a) + len(b) - 2)
    )
    d = float((b.mean() - a.mean()) / pooled)
    df = len(a) + len(b) - 2
    correction = 1 - 3 / (4 * df - 1)
    return d, d * correction

def welch_difference_ci(group1: np.ndarray, group2: np.ndarray) -> tuple[float, float, float]:
    a, b = np.asarray(group1, float), np.asarray(group2, float)
    difference = float(b.mean() - a.mean())
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    se = math.sqrt(va + vb)
    df = (va + vb) ** 2 / (va**2 / (len(a) - 1) + vb**2 / (len(b) - 1))
    critical = stats.t.ppf(0.975, df)
    return difference, difference - critical * se, difference + critical * se

## Reading and checking the survey results

In [14]:
def prepare_data(path: Path) -> tuple[pd.DataFrame, dict[str, str], pd.DataFrame]:
    raw = pd.read_excel(path)
    columns = list(raw.columns)
    group_col = find_column(columns, "Group")
    id_col = find_column(columns, "ID")
    domain_col = find_column(columns, "Domain")
    consent_col = find_column(columns, CONSENT)
    start_col = find_column(columns, "Start time")
    completion_col = find_column(columns, "Completion time")

    consent = raw[consent_col].map(normalize).eq("yes")
    data = raw.loc[consent].copy()
    data["Group"] = pd.to_numeric(data[group_col], errors="raise").astype(int)
    data["Participant_ID"] = data[id_col].astype(str)
    data["Domain"] = pd.to_numeric(data[domain_col], errors="raise").astype(int)
    data["Domain_name"] = data["Domain"].map(DOMAIN_NAMES)
    if data["Domain_name"].isna().any():
        raise ValueError("Domain must be coded 1, 2, or 3.")
    data["Start_time"] = pd.to_datetime(data[start_col], errors="coerce")
    data["Completion_time"] = pd.to_datetime(data[completion_col], errors="coerce")
    data["Duration_minutes"] = (
        (data["Completion_time"] - data["Start_time"]).dt.total_seconds() / 60
    )

    item_map: dict[str, str] = {}
    for dimension, questions in ITEMS.items():
        for index, question in enumerate(questions, 1):
            aliases: tuple[str, ...] = ()
            if dimension == "Clarity" and index == 2:
                aliases = ("I did not need any clarification to interpret the explanation.",)
            if dimension == "Trust" and index == 3:
                aliases = ("I feel secure relying on the LLM tool for correct explanations.",)
            source = find_column(columns, question, aliases)
            target = f"{dimension}_{index}"
            item_map[target] = source
            data[target] = pd.to_numeric(data[source], errors="coerce")
            invalid = ~data[target].isin(range(1, 8))
            if invalid.any():
                raise ValueError(f"{source}: expected complete numeric responses from 1 through 7.")

    for dimension in ITEMS:
        data[dimension] = data[[f"{dimension}_{i}" for i in range(1, 4)]].mean(axis=1)
    for construct, dimensions in CONSTRUCTS.items():
        data[construct] = data[dimensions].mean(axis=1)

    # Verify IBM's supplied calculated columns without relying on them.
    score_checks = []
    normalized_columns = {normalize(column): column for column in columns}
    for scale in [*ITEMS.keys(), *CONSTRUCTS.keys()]:
        supplied_col = normalized_columns.get(normalize(scale))
        if supplied_col is None:
            continue
        supplied = pd.to_numeric(data[supplied_col], errors="coerce")
        difference = (data[scale] - supplied).abs()
        score_checks.append(
            {
                "Scale": scale,
                "n_compared": int(difference.notna().sum()),
                "maximum_absolute_difference": float(difference.max()),
                "mean_absolute_difference": float(difference.mean()),
            }
        )
    return data, item_map, pd.DataFrame(score_checks)

In [15]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f'Update DATA_FILE. Could not find: {DATA_FILE.resolve()}')

data_all, item_map, score_checks = prepare_data(DATA_FILE)
participant_count = data_all[['Group', 'Participant_ID']].drop_duplicates().shape[0]
print(f'{len(data_all)} domain rows from {participant_count} consenting participants')

147 domain rows from 49 consenting participants


In [16]:
score_checks.round(6)

,Scale,n_compared,maximum_absolute_difference,mean_absolute_difference
0,Completeness,147,0.0,0.0
1,Soundness,147,0.0,0.0
2,Causability,147,0.0,0.0
3,Clarity,147,0.0,0.0
4,Compactness,147,0.0,0.0
5,Comprehensibility,147,0.0,0.0
6,Curiosity,147,0.0,0.0
7,Trust,147,0.0,0.0
8,Fidelity,147,0.0,0.0
9,Interpretability,147,0.0,0.0


## Looking for the outliers

In [17]:
def participant_audit(data: pd.DataFrame) -> pd.DataFrame:
    item_columns = [f"{dimension}_{i}" for dimension in ITEMS for i in range(1, 4)]
    rows = []
    for (group, participant), subset in data.groupby(["Group", "Participant_ID"]):
        responses = subset[item_columns].to_numpy(float).ravel()
        rows.append(
            {
                "Group": int(group),
                "Participant_ID": participant,
                "Domains_completed": int(subset["Domain"].nunique()),
                "Response_variance_across_72_items": float(responses.var(ddof=1)),
                "Mean_response": float(responses.mean()),
                "Duration_minutes": float(subset["Duration_minutes"].iloc[0]),
                "Missing_item_responses": int(np.isnan(responses).sum()),
            }
        )
    audit = pd.DataFrame(rows).sort_values("Response_variance_across_72_items")
    audit["Low_variance_rank"] = np.arange(1, len(audit) + 1)
    return audit

In [18]:
def construct_outlier_audit(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for domain, domain_data in data.groupby("Domain_name"):
        for construct in CONSTRUCTS:
            q1, q3 = domain_data[construct].quantile([0.25, 0.75])
            iqr = q3 - q1
            low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            flagged = domain_data.loc[
                (domain_data[construct] < low) | (domain_data[construct] > high)
            ]
            for row in flagged.itertuples():
                rows.append(
                    {
                        "Domain": domain,
                        "Construct": construct,
                        "Group": row.Group,
                        "Participant_ID": row.Participant_ID,
                        "Score": getattr(row, construct),
                        "Lower_fence": low,
                        "Upper_fence": high,
                    }
                )
    return pd.DataFrame(rows)

In [19]:
def select_main_sample(
    data: pd.DataFrame,
    audit: pd.DataFrame,
    exclusions: list[tuple[int, str]] | None,
    keep_all: bool,
) -> tuple[pd.DataFrame, list[tuple[int, str]], str]:
    if keep_all:
        selected: list[tuple[int, str]] = []
        method = "No respondents excluded (--keep-all-main)."
    elif exclusions:
        selected = exclusions
        method = "User-specified exclusions."
    else:
        # The paper states that two mildly low-variance respondents were removed,
        # but it does not give their IDs or an exact numerical cutoff. Selecting the
        # two lowest-variance respondents is therefore an explicit reconstruction,
        # not a claim that the unpublished original exclusion rule is known.
        selected = [
            (int(row.Group), str(row.Participant_ID))
            for row in audit.nsmallest(2, "Response_variance_across_72_items").itertuples()
        ]
        method = (
            "Inferred reconstruction: excluded the two respondents with the lowest "
            "variance across all 72 ratings, because the paper reports two such exclusions."
        )
    keys = set(selected)
    keep = [
        (int(group), str(participant)) not in keys
        for group, participant in zip(data["Group"], data["Participant_ID"])
    ]
    return data.loc[keep].copy(), selected, method

In [20]:
participant_checks = participant_audit(data_all)
construct_outliers = construct_outlier_audit(data_all)
data_main, selected_exclusions, exclusion_method = select_main_sample(
    data_all, participant_checks, EXCLUSIONS, KEEP_ALL_PARTICIPANTS
)

main_count = data_main[['Group', 'Participant_ID']].drop_duplicates().shape[0]
print(f'Participants used for the main models: {main_count}')
print(f'Excluded: {selected_exclusions}')
print(exclusion_method)

Participants used for the main models: 47
Excluded: [(1, '14'), (2, '46')]
Inferred reconstruction: excluded the two respondents with the lowest variance across all 72 ratings, because the paper reports two such exclusions.


In [21]:
participant_checks.head(10).round(3)

,Group,Participant_ID,Domains_completed,Response_variance_across_72_items,Mean_response,Duration_minutes,Missing_item_responses,Low_variance_rank
5,1,14,3,0.338,1.514,5.250,0,1
45,2,46,3,0.972,5.986,2.933,0,2
8,1,17,3,1.159,4.347,30.367,0,3
38,2,39,3,1.688,3.542,175.333,0,4
10,1,19,3,1.773,3.208,47.900,0,5
35,2,36,3,1.783,3.361,50.517,0,6
31,2,32,3,1.840,3.139,88.683,0,7
28,2,29,3,1.887,4.667,18.033,0,8
27,2,28,3,1.912,2.444,10.450,0,9
9,1,18,3,1.969,3.556,11.200,0,10


In [22]:
construct_outliers.round(3) if not construct_outliers.empty else pd.DataFrame({'Result': ['No construct-level outliers']})

,Domain,Construct,Group,Participant_ID,Score,Lower_fence,Upper_fence
0,Loan approval,Fidelity,1,11,7.000,0.667,6.889
1,Loan approval,Fidelity,2,46,7.000,0.667,6.889
2,Parking fines,Interpretability,2,46,5.778,0.611,5.500


## Reliability analysis

In [23]:
def reliability_table(data: pd.DataFrame, sample_label: str) -> pd.DataFrame:
    rows = []
    for dimension in ITEMS:
        columns = [f"{dimension}_{i}" for i in range(1, 4)]
        rows.append(
            {
                "Sample": sample_label,
                "Scale": dimension,
                "Method": "three raw items",
                "Cronbach_alpha": cronbach_alpha(data[columns]),
            }
        )
    rows.append(
        {
            "Sample": sample_label,
            "Scale": "Compactness_without_item_3",
            "Method": "two raw items",
            "Cronbach_alpha": cronbach_alpha(data[["Compactness_1", "Compactness_2"]]),
        }
    )
    for construct, dimensions in CONSTRUCTS.items():
        rows.append(
            {
                "Sample": sample_label,
                "Scale": construct,
                "Method": "three dimension scores (paper-comparable)",
                "Cronbach_alpha": cronbach_alpha(data[dimensions]),
            }
        )
        nine_items = [f"{dimension}_{i}" for dimension in dimensions for i in range(1, 4)]
        rows.append(
            {
                "Sample": sample_label,
                "Scale": construct,
                "Method": "nine raw items (sensitivity analysis)",
                "Cronbach_alpha": cronbach_alpha(data[nine_items]),
            }
        )
    return pd.DataFrame(rows)

In [24]:
reliability = pd.concat(
    [
        reliability_table(data_all, 'All 49 consenting participants'),
        reliability_table(data_main, 'Final sample (47)'),
    ],
    ignore_index=True,
)
reliability.round(3)

,Sample,Scale,Method,Cronbach_alpha
0,All 49 consenting participants,Completeness,three raw items,0.908
1,All 49 consenting participants,Soundness,three raw items,0.626
2,All 49 consenting participants,Causability,three raw items,0.685
3,All 49 consenting participants,Clarity,three raw items,0.807
4,All 49 consenting participants,Compactness,three raw items,0.494
5,All 49 consenting participants,Comprehensibility,three raw items,0.644
6,All 49 consenting participants,Curiosity,three raw items,0.917
7,All 49 consenting participants,Trust,three raw items,0.896
8,All 49 consenting participants,Compactness_without_item_3,two raw items,0.516
9,All 49 consenting participants,Fidelity,three dimension scores (paper-comparable),0.805


## PCA (one-component)

In [25]:
def pca_table(data: pd.DataFrame, sample_label: str) -> pd.DataFrame:
    rows = []
    for dimension in ITEMS:
        columns = [f"{dimension}_{i}" for i in range(1, 4)]
        loadings, explained = pca_one_component(data[columns])
        rows.append(
            {
                "Sample": sample_label,
                "Dimension": dimension,
                "Item_1": loadings[0],
                "Item_2": loadings[1],
                "Item_3": loadings[2],
                "Variance_explained_percent": explained,
            }
        )
    return pd.DataFrame(rows)

In [27]:
pca_results = pd.concat(
    [
        pca_table(data_all, 'All 49 consenting participants'),
        pca_table(data_main, 'Final sample (47)'),
    ],
    ignore_index=True,
)
pca_results.round(3)

,Sample,Dimension,Item_1,Item_2,Item_3,Variance_explained_percent
0,All 49 consenting participants,Completeness,0.919,0.924,0.914,84.422
1,All 49 consenting participants,Soundness,0.837,0.606,0.821,58.058
2,All 49 consenting participants,Causability,0.829,0.804,0.718,61.619
3,All 49 consenting participants,Clarity,0.846,0.863,0.841,72.243
4,All 49 consenting participants,Compactness,0.714,0.804,0.584,49.887
5,All 49 consenting participants,Comprehensibility,0.803,0.667,0.835,59.538
6,All 49 consenting participants,Curiosity,0.930,0.955,0.894,85.883
7,All 49 consenting participants,Trust,0.934,0.869,0.928,82.948
8,Final sample (47),Completeness,0.915,0.918,0.906,83.356
9,Final sample (47),Soundness,0.834,0.527,0.817,54.689


## General statistics

In [28]:
def descriptives_table(data: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (domain, group), subset in data.groupby(["Domain_name", "Group"], sort=False):
        for construct, dimensions in CONSTRUCTS.items():
            for dimension in dimensions:
                rows.append(
                    {
                        "Domain": domain,
                        "Construct": construct,
                        "Dimension": dimension,
                        "Group": int(group),
                        "M": subset[dimension].mean(),
                        "SD": subset[dimension].std(ddof=1),
                        "SE": subset[dimension].sem(ddof=1),
                        "n": len(subset),
                    }
                )
    return pd.DataFrame(rows)

In [29]:
descriptives = descriptives_table(data_main)
descriptives.round(3)

,Domain,Construct,Dimension,Group,M,SD,SE,n
0,Pizza delivery,Fidelity,Completeness,1,4.486,1.582,0.323,24
1,Pizza delivery,Fidelity,Soundness,1,4.264,1.224,0.250,24
2,Pizza delivery,Fidelity,Causability,1,4.583,1.435,0.293,24
3,Pizza delivery,Interpretability,Clarity,1,4.375,1.201,0.245,24
4,Pizza delivery,Interpretability,Compactness,1,3.500,1.138,0.232,24
5,Pizza delivery,Interpretability,Comprehensibility,1,4.083,1.294,0.264,24
6,Pizza delivery,Fidelity,Completeness,2,3.739,1.928,0.402,23
7,Pizza delivery,Fidelity,Soundness,2,3.783,1.489,0.311,23
8,Pizza delivery,Fidelity,Causability,2,4.058,1.340,0.279,23
9,Pizza delivery,Interpretability,Clarity,2,4.696,1.428,0.298,23


## ANOVA, ANCOVA

In [30]:
def model_tables(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    model_rows, term_rows = [], []
    for domain, subset in data.groupby("Domain_name", sort=False):
        group = (subset["Group"] == 2).to_numpy(float)
        centered_covariates = subset[["Curiosity", "Trust"]] - subset[["Curiosity", "Trust"]].mean()
        for outcome in CONSTRUCTS:
            y = subset[outcome].to_numpy(float)
            raw_g1 = subset.loc[subset["Group"] == 1, outcome].to_numpy(float)
            raw_g2 = subset.loc[subset["Group"] == 2, outcome].to_numpy(float)
            difference, ci_low, ci_high = welch_difference_ci(raw_g1, raw_g2)
            d, g = hedges_g(raw_g1, raw_g2)
            for model_name in ("ANOVA", "ANCOVA"):
                if model_name == "ANOVA":
                    x = np.column_stack([np.ones(len(subset)), group])
                    names = ["Intercept", "Group_2_minus_1"]
                else:
                    x = np.column_stack([np.ones(len(subset)), group, centered_covariates])
                    names = ["Intercept", "Group_2_minus_1", "Curiosity", "Trust"]
                model = fit_ols(y, x)
                if model_name == "ANOVA":
                    means = {
                        1: (raw_g1.mean(), stats.sem(raw_g1)),
                        2: (raw_g2.mean(), stats.sem(raw_g2)),
                    }
                else:
                    means = {}
                    beta = np.asarray(model["beta"])
                    covariance = np.asarray(model["covariance"])
                    for g_code in (0, 1):
                        contrast = np.array([1.0, float(g_code), 0.0, 0.0])
                        means[g_code + 1] = (
                            float(contrast @ beta),
                            math.sqrt(float(contrast @ covariance @ contrast)),
                        )
                model_rows.append(
                    {
                        "Domain": domain,
                        "Construct": outcome,
                        "Model": model_name,
                        "Group_1_M": means[1][0],
                        "Group_1_SE": means[1][1],
                        "Group_2_M": means[2][0],
                        "Group_2_SE": means[2][1],
                        "df_model": model["df_model"],
                        "df_residual": model["df_resid"],
                        "Whole_model_F": model["F"],
                        "Whole_model_p": model["p"],
                        "Whole_model_eta_squared": model["R2_eta_squared"],
                        "Raw_G2_minus_G1": difference,
                        "Raw_difference_CI95_low": ci_low,
                        "Raw_difference_CI95_high": ci_high,
                        "Cohen_d": d,
                        "Hedges_g": g,
                    }
                )
                for index, name in enumerate(names[1:], 1):
                    result = coefficient_test(model, index)
                    term_rows.append(
                        {
                            "Domain": domain,
                            "Construct": outcome,
                            "Model": model_name,
                            "Term": name,
                            "df1": 1,
                            "df2": model["df_resid"],
                            **result,
                        }
                    )
    return pd.DataFrame(model_rows), pd.DataFrame(term_rows)

In [31]:
model_results, model_terms = model_tables(data_main)
model_results[[
    'Domain', 'Construct', 'Model', 'Group_1_M', 'Group_1_SE',
    'Group_2_M', 'Group_2_SE', 'df_model', 'df_residual',
    'Whole_model_F', 'Whole_model_p', 'Whole_model_eta_squared',
    'Raw_G2_minus_G1', 'Raw_difference_CI95_low',
    'Raw_difference_CI95_high', 'Cohen_d'
]].round(4)

,Domain,Construct,Model,Group_1_M,Group_1_SE,Group_2_M,Group_2_SE,df_model,df_residual,Whole_model_F,Whole_model_p,Whole_model_eta_squared,Raw_G2_minus_G1,Raw_difference_CI95_low,Raw_difference_CI95_high,Cohen_d
0,Pizza delivery,Fidelity,ANOVA,4.4444,0.2389,3.8599,0.2919,1,45,2.4200,0.1268,0.0510,-0.5845,-1.3453,0.1762,-0.4539
1,Pizza delivery,Fidelity,ANCOVA,4.4612,0.2244,3.8425,0.2293,3,43,7.6092,0.0003,0.3468,-0.5845,-1.3453,0.1762,-0.4539
2,Pizza delivery,Interpretability,ANOVA,3.9861,0.1641,4.1691,0.2314,1,45,0.4217,0.5194,0.0093,0.1830,-0.3903,0.7563,0.1895
3,Pizza delivery,Interpretability,ANCOVA,4.0155,0.1751,4.1385,0.1789,3,43,5.0799,0.0042,0.2617,0.1830,-0.3903,0.7563,0.1895
4,Parking fines,Fidelity,ANOVA,3.2315,0.1961,3.7198,0.2233,1,45,2.7124,0.1065,0.0568,0.4883,-0.1106,1.0873,0.4806
5,Parking fines,Fidelity,ANCOVA,3.3189,0.1981,3.6286,0.2025,3,43,3.6764,0.0192,0.2041,0.4883,-0.1106,1.0873,0.4806
6,Parking fines,Interpretability,ANOVA,3.6343,0.1996,2.5652,0.1859,1,45,15.3043,0.0003,0.2538,-1.0690,-1.6184,-0.5197,-1.1415
7,Parking fines,Interpretability,ANCOVA,3.7389,0.1771,2.4560,0.1811,3,43,9.8440,0.0000,0.4072,-1.0690,-1.6184,-0.5197,-1.1415
8,Loan approval,Fidelity,ANOVA,4.0509,0.2244,3.6280,0.1988,1,45,1.9778,0.1665,0.0421,-0.4229,-1.0269,0.1811,-0.4104
9,Loan approval,Fidelity,ANCOVA,4.0221,0.1456,3.6581,0.1488,3,43,19.1693,0.0000,0.5722,-0.4229,-1.0269,0.1811,-0.4104


In [32]:
model_terms[[
    'Domain', 'Construct', 'Model', 'Term', 'coefficient', 'SE',
    'F', 'p', 'partial_eta_squared'
]].round(4)

,Domain,Construct,Model,Term,coefficient,SE,F,p,partial_eta_squared
0,Pizza delivery,Fidelity,ANOVA,Group_2_minus_1,-0.5845,0.3758,2.4200,0.1268,0.0510
1,Pizza delivery,Fidelity,ANCOVA,Group_2_minus_1,-0.6187,0.3228,3.6735,0.0619,0.0787
2,Pizza delivery,Fidelity,ANCOVA,Curiosity,0.0705,0.1087,0.4211,0.5198,0.0097
3,Pizza delivery,Fidelity,ANCOVA,Trust,0.5112,0.1212,17.7904,0.0001,0.2927
4,Pizza delivery,Interpretability,ANOVA,Group_2_minus_1,0.1830,0.2818,0.4217,0.5194,0.0093
5,Pizza delivery,Interpretability,ANCOVA,Group_2_minus_1,0.1230,0.2518,0.2386,0.6277,0.0055
6,Pizza delivery,Interpretability,ANCOVA,Curiosity,-0.0357,0.0848,0.1773,0.6758,0.0041
7,Pizza delivery,Interpretability,ANCOVA,Trust,0.3622,0.0946,14.6750,0.0004,0.2544
8,Parking fines,Fidelity,ANOVA,Group_2_minus_1,0.4883,0.2965,2.7124,0.1065,0.0568
9,Parking fines,Fidelity,ANCOVA,Group_2_minus_1,0.3097,0.2878,1.1585,0.2878,0.0262


Parking fines (ANCOVA)

In [33]:
parking_terms = model_terms.query(
    "Domain == 'Parking fines' and Model == 'ANCOVA'"
)
parking_terms[[
    'Construct', 'Term', 'coefficient', 'SE', 'F', 'p',
    'partial_eta_squared'
]].round(4)

,Construct,Term,coefficient,SE,F,p,partial_eta_squared
9,Fidelity,Group_2_minus_1,0.3097,0.2878,1.1585,0.2878,0.0262
10,Fidelity,Curiosity,-0.1190,0.0978,1.4805,0.2303,0.0333
11,Fidelity,Trust,0.3184,0.1203,7.0021,0.0113,0.1400
13,Interpretability,Group_2_minus_1,-1.2829,0.2573,24.8545,0.0000,0.3663
14,Interpretability,Curiosity,-0.2219,0.0875,6.4333,0.0149,0.1301
15,Interpretability,Trust,0.2572,0.1076,5.7122,0.0213,0.1173


## Comparing against paper stated values

In [34]:
def paper_comparison(
    reliability: pd.DataFrame, pca: pd.DataFrame, models: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    alpha_rows = []
    for scale, expected in PAPER_ALPHA.items():
        current = reliability.loc[
            (reliability["Sample"] == "All 49 consenting participants")
            & (reliability["Scale"] == scale)
        ]
        if scale in CONSTRUCTS:
            current = current.loc[
                current["Method"] == "three dimension scores (paper-comparable)"
            ]
        observed = float(current.iloc[0]["Cronbach_alpha"])
        alpha_rows.append(
            {
                "Scale": scale,
                "Paper": expected,
                "Reconstructed": observed,
                "Difference": observed - expected,
                "Matches_rounding_tolerance": abs(observed - expected) <= 0.015,
            }
        )

    pca_rows = []
    pooled = pca.loc[pca["Sample"] == "All 49 consenting participants"]
    for dimension, (expected_loadings, expected_variance) in PAPER_PCA.items():
        observed = pooled.loc[pooled["Dimension"] == dimension].iloc[0]
        for index, expected in enumerate(expected_loadings, 1):
            value = float(observed[f"Item_{index}"])
            pca_rows.append(
                {
                    "Dimension": dimension,
                    "Metric": f"Item_{index}_loading",
                    "Paper": expected,
                    "Reconstructed": value,
                    "Difference": value - expected,
                }
            )
        value = float(observed["Variance_explained_percent"])
        pca_rows.append(
            {
                "Dimension": dimension,
                "Metric": "Variance_explained_percent",
                "Paper": expected_variance,
                "Reconstructed": value,
                "Difference": value - expected_variance,
            }
        )

    model_rows = []
    for key, benchmark in PAPER_MODELS.items():
        domain, construct, model_name = key
        observed = models.loc[
            (models["Domain"] == domain)
            & (models["Construct"] == construct)
            & (models["Model"] == model_name)
        ].iloc[0]
        paper_g1, paper_g2, paper_f, paper_p, paper_eta = benchmark
        model_rows.append(
            {
                "Domain": domain,
                "Construct": construct,
                "Model": model_name,
                "Paper_Group_1_M": paper_g1,
                "Reconstructed_Group_1_M": observed["Group_1_M"],
                "Paper_Group_2_M": paper_g2,
                "Reconstructed_Group_2_M": observed["Group_2_M"],
                "Paper_F": paper_f,
                "Reconstructed_F": observed["Whole_model_F"],
                "Paper_p": paper_p,
                "Reconstructed_p": observed["Whole_model_p"],
                "Paper_eta_squared": paper_eta,
                "Reconstructed_eta_squared": observed["Whole_model_eta_squared"],
                "Correct_reconstructed_df": (
                    f"({int(observed['df_model'])},{int(observed['df_residual'])})"
                ),
            }
        )
    return pd.DataFrame(alpha_rows), pd.DataFrame(pca_rows), pd.DataFrame(model_rows)

In [35]:
def covariate_term_comparison(model_terms: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (domain, construct, term), (paper_f, paper_p) in PAPER_COVARIATE_TERMS.items():
        observed = model_terms.loc[
            (model_terms["Domain"] == domain)
            & (model_terms["Construct"] == construct)
            & (model_terms["Model"] == "ANCOVA")
            & (model_terms["Term"] == term)
        ].iloc[0]
        rows.append(
            {
                "Domain": domain,
                "Construct": construct,
                "Term": term,
                "Paper_F": paper_f,
                "Reconstructed_F": observed["F"],
                "Difference": observed["F"] - paper_f,
                "Paper_p": paper_p,
                "Reconstructed_p": observed["p"],
                "Possible_reporting_issue": abs(observed["F"] - paper_f) > 0.05,
            }
        )
    return pd.DataFrame(rows)

In [41]:
alpha_comparison, pca_comparison, model_comparison = paper_comparison(
    reliability, pca_results, model_results
)
covariate_comparison = covariate_term_comparison(model_terms)

In [42]:
# Reliability
alpha_comparison.round(4)

,Scale,Paper,Reconstructed,Difference,Matches_rounding_tolerance
0,Completeness,0.910,0.9077,-0.0023,True
1,Soundness,0.630,0.6264,-0.0036,True
2,Causability,0.690,0.6848,-0.0052,True
3,Clarity,0.810,0.8073,-0.0027,True
4,Compactness,0.490,0.4937,0.0037,True
5,Comprehensibility,0.640,0.6438,0.0038,True
6,Curiosity,0.920,0.9167,-0.0033,True
7,Trust,0.900,0.8964,-0.0036,True
8,Fidelity,0.810,0.8046,-0.0054,True
9,Interpretability,0.800,0.8040,0.0040,True


In [43]:
# PCA
pca_comparison.round(4)

,Dimension,Metric,Paper,Reconstructed,Difference
0,Completeness,Item_1_loading,0.919,0.9189,-0.0001
1,Completeness,Item_2_loading,0.924,0.9240,-0.0000
2,Completeness,Item_3_loading,0.914,0.9135,-0.0005
3,Completeness,Variance_explained_percent,84.420,84.4225,0.0025
4,Soundness,Item_1_loading,0.837,0.8367,-0.0003
5,Soundness,Item_2_loading,0.606,0.6058,-0.0002
6,Soundness,Item_3_loading,0.821,0.8214,0.0004
7,Soundness,Variance_explained_percent,58.050,58.0575,0.0075
8,Causability,Item_1_loading,0.829,0.8287,-0.0003
9,Causability,Item_2_loading,0.804,0.8043,0.0003


In [44]:
# ANOVA, ANCOVA
model_comparison.round(4)

,Domain,Construct,Model,Paper_Group_1_M,Reconstructed_Group_1_M,Paper_Group_2_M,Reconstructed_Group_2_M,Paper_F,Reconstructed_F,Paper_p,Reconstructed_p,Paper_eta_squared,Reconstructed_eta_squared,Correct_reconstructed_df
0,Pizza delivery,Fidelity,ANOVA,4.44,4.4444,3.86,3.8599,2.420,2.4200,0.12,0.1268,0.051,0.0510,"(1,45)"
1,Pizza delivery,Interpretability,ANOVA,3.98,3.9861,4.16,4.1691,0.420,0.4217,0.51,0.5194,0.009,0.0093,"(1,45)"
2,Pizza delivery,Fidelity,ANCOVA,4.46,4.4612,3.84,3.8425,7.600,7.6092,<.001,0.0003,0.347,0.3468,"(3,43)"
3,Pizza delivery,Interpretability,ANCOVA,4.01,4.0155,4.13,4.1385,5.080,5.0799,0.004,0.0042,0.262,0.2617,"(3,43)"
4,Parking fines,Fidelity,ANOVA,3.23,3.2315,3.71,3.7198,2.710,2.7124,0.107,0.1065,0.057,0.0568,"(1,45)"
5,Parking fines,Interpretability,ANOVA,3.63,3.6343,2.56,2.5652,15.300,15.3043,<.001,0.0003,0.254,0.2538,"(1,45)"
6,Parking fines,Fidelity,ANCOVA,3.31,3.3189,3.62,3.6286,3.670,3.6764,0.019,0.0192,0.204,0.2041,"(3,43)"
7,Parking fines,Interpretability,ANCOVA,3.73,3.7389,2.45,2.4560,9.840,9.8440,<.001,0.0000,0.407,0.4072,"(3,43)"
8,Loan approval,Fidelity,ANOVA,4.05,4.0509,3.62,3.6280,1.970,1.9778,0.16,0.1665,0.042,0.0421,"(1,45)"
9,Loan approval,Interpretability,ANOVA,3.31,3.3287,3.25,3.2512,0.058,0.0577,0.81,0.8113,0.001,0.0013,"(1,45)"


In [45]:
# Covariate statistics
covariate_comparison.round(4)

,Domain,Construct,Term,Paper_F,Reconstructed_F,Difference,Paper_p,Reconstructed_p,Possible_reporting_issue
0,Pizza delivery,Fidelity,Trust,17.79,17.7904,0.0004,<.001,0.0001,False
1,Pizza delivery,Interpretability,Trust,14.67,14.6750,0.0050,<.001,0.0004,False
2,Parking fines,Fidelity,Trust,5.71,7.0021,1.2921,0.021,0.0113,True
3,Parking fines,Interpretability,Curiosity,6.43,6.4333,0.0033,0.015,0.0149,False
4,Parking fines,Interpretability,Trust,7.00,5.7122,-1.2878,0.011,0.0213,True
5,Loan approval,Fidelity,Curiosity,6.67,6.6744,0.0044,0.013,0.0133,False
6,Loan approval,Fidelity,Trust,51.40,51.4025,0.0025,<.001,0.0000,False
7,Loan approval,Interpretability,Trust,31.58,31.5825,0.0025,<.001,0.0000,False


## Additional check

In [46]:
significant_group_terms = model_terms.query(
    "Model == 'ANCOVA' and Term == 'Group_2_minus_1' and p < 0.05"
)[["Domain", "Construct", "coefficient", "p"]]

print('Significant adjusted group terms:')
significant_group_terms.round(5)

Significant adjusted group terms:


,Domain,Construct,coefficient,p
13,Parking fines,Interpretability,-1.28294,0.00001
